In [1]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor,cub
import tensorly as tl
import plotly.io as pio
pio.renderers.default = 'iframe'
tl.set_backend('cupy')
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()
print(f'TensorLy backend: {tl.get_backend()}')

env: CUPY_ACCELERATORS=cutensor,cub


TensorLy backend: cupy


In [3]:
from moabb.datasets import *
from notebooks.data_loader import load_moabb_p300

epochs, labels, meta = load_moabb_p300(BI2012, subjects=[2], session=0)
epochs

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 764 events (all good), 0 – 1 s (baseline off), ~12.1 MB, data loaded,
 'Target': 128
 'NonTarget': 636>



Adding metadata with 3 columns
Adding metadata with 3 columns
764 matching events found
No baseline correction applied


/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/paradigms/base.py:350: RuntimeWarning:

Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.



<EpochsArray | 764 events (all good), 0 – 0.979 s (baseline off), ~4.5 MB, data loaded, with metadata,
 'Target': 128
 'NonTarget': 636>

In [4]:
from mne import combine_evoked

contrast = combine_evoked([epochs['Target'].average(), epochs['NonTarget'].average()], weights=[1,-1])
contrast.plot_joint();

No projector specified for this dataset. Please consider the method self.add_proj.


In [5]:
from notebooks.data_loader import postprocess
import cupy
X,y = postprocess(epochs, labels)
X.shape

(1, 16, 1)


(764, 16, 48)

In [6]:
from hoda.hoda import HODA

hoda = HODA(
        rank=4,
        max_iter=64,
        tol=1e-6,
        shrinkage='lw',
        refit_shrinkage=False,
        obj='tr',
        solver='lanczos',
        toeplitz=None,
        taper=False,
        extra_train_info=False,
        verbose=False,
        theta=None,
        forward=True       
)


In [7]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from hoda.util import flip_signs
from hoda.cov import mode_scatter, ledoit_wolf_shrinkage
from hoda.util import solve_gevdh

plt.style.use('default')

%load_ext line_profiler
hoda.fit_backward(X,y)
df = pd.DataFrame(hoda.train_info_['backward'])

In [8]:
0.001231992088601549
0.05504274022049101

0.05504274022049101

In [9]:
df

,iteration,mode,flip,update,shrinkage,objective
0,1,1,1,1.492175e+00,0.040250,160082.700959
1,1,2,2,1.301742e+00,0.027264,77186.252117
2,2,1,3,1.123068e+00,0.040250,109380.330469
3,2,2,4,1.269723e+00,0.027264,35869.911632
4,3,1,5,4.802877e-01,0.040250,74172.084164
...,...,...,...,...,...,...
63,32,2,64,7.100333e-07,0.027264,17731.160889
64,33,1,65,1.118562e-06,0.040250,58559.827797
65,33,2,66,5.053964e-07,0.027264,17731.160024
66,34,1,67,7.961845e-07,0.040250,58559.822762


In [10]:
import plotly.express as px

if hoda.extra_train_info:
    fig = px.line(df, x='flip', y='F_tr')
    fig.show()


In [11]:
if hoda.extra_train_info:
    fig = px.line(df, x='flip', y='F_rt')
    fig.show()


In [12]:

px.line(df, x='flip', y='update', log_y=True, color='mode')


In [13]:
px.line(df, x='flip', y='shrinkage', log_y=False, color='mode')

In [14]:
px.line(df, x='flip', y='objective', log_y=True, color='mode')

In [15]:
from hoda.util import ridge_regression

%lprun -f ridge_regression hoda.fit_forward(X,y)
df = pd.DataFrame(hoda.train_info_['forward'])
df

,iteration,mode,flip,update,lambda_
0,1,1,1,6.310812e-01,0.0
1,1,2,2,9.790931e-01,0.0
2,2,1,3,3.536537e-01,0.0
3,2,2,4,2.375119e-01,0.0
4,3,1,5,1.046053e-01,0.0
5,3,2,6,9.806495e-02,0.0
6,4,1,7,3.853421e-02,0.0
7,4,2,8,4.211184e-02,0.0
8,5,1,9,1.552617e-02,0.0
9,5,2,10,1.847504e-02,0.0


Timer unit: 1e-09 s

Total time: 0.0229769 s
File: /vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/util.py
Function: ridge_regression at line 117

Line #      Hits         Time  Per Hit   % Time  Line Contents
   117                                           def ridge_regression(X, Y, lambda_=0):
   118                                                   """
   119                                                   Compute the ridge regression solution for matrix Y.
   120                                                   
   121                                                   X: Input matrix (n x p)
   122                                                   Y: Target matrix (n x m)
   123                                                   lambda_: Regularization parameter
   124                                                   
   125                                                   Returns:
   126                                                   W: The ridge regression wei

In [16]:
if hoda.extra_train_info:
    px.line(df, x='flip', y='mse', log_y=True)

In [17]:
px.line(df, x='flip', y='update', log_y=True, color='mode')

In [18]:
px.line(df, x='flip', y='lambda_', log_y=True, color='mode')

In [19]:
import numpy as np
from mne.viz import plot_topomap
import matplotlib.pyplot as plt
import tensorly as tl

col_wrap = 4

n_col = col_wrap
n_row = int( np.ceil(hoda.rank_[0] / col_wrap)) 
fig, axs = plt.subplots(n_row, n_col, layout='tight')
vmax = float(tl.max(tl.abs(hoda.weights_[0])))
for i in range(hoda.rank_[0]):
    w = tl.to_numpy(hoda.weights_[0][:,i])
    plot_topomap(w, epochs.info, axes=axs.flatten()[i], show=False, vlim=[-vmax, vmax])

In [20]:
from mne.viz import plot_topomap
from plotly.tools import mpl_to_plotly

# TODO spatial weights
df = pd.DataFrame(tl.to_numpy(hoda.weights_[1]))
df['time'] = epochs.times
df = df.melt(id_vars=['time'], var_name='rank', value_name='weight')
px.line(df, x='time', y='weight', facet_col='rank', facet_col_wrap=4)

In [21]:

n_row = int(np.ceil(hoda.rank_[0] / col_wrap)) 
fig, axs = plt.subplots(n_row, n_col, layout='tight')
A = hoda.aps_[0]
vmax = float(tl.max(tl.abs(A)))
for i in range(hoda.rank_[0]):
    a = tl.to_numpy(A[:,i])
    plot_topomap(a, epochs.info, axes=axs.flatten()[i], show=False, vlim=[-vmax, vmax])


In [22]:
df = pd.DataFrame(tl.to_numpy(hoda.aps_[1]))
df['time'] = epochs.times
df = df.melt(id_vars=['time'], var_name='rank', value_name='amplitude')
px.line(df, x='time', y='amplitude', facet_col='rank', facet_col_wrap=4)

In [23]:
from sklearn.feature_selection import SelectFpr, SelectFwe, SelectFdr
import numpy as np
from sklearn.preprocessing import StandardScaler

Xt = hoda.transform(X)
xt = tl.to_numpy(tl.unfold(Xt,0))
xt = StandardScaler().fit_transform(xt)

select = SelectFdr(alpha=.05)
xts = select.fit_transform(xt,y)

df = pd.DataFrame({
    'feature': np.arange(xt.shape[-1]),
    'F': select.scores_,
    'p_value': select.pvalues_,
    'significant': select.get_support()
})
df

,feature,F,p_value,significant
0,0,0.172135,6.783383e-01,False
1,1,1.200217,2.736245e-01,False
2,2,0.039084,8.433350e-01,False
3,3,2.369173,1.241677e-01,False
4,4,0.030377,8.616837e-01,False
5,5,0.308776,5.785952e-01,False
6,6,0.360919,5.481749e-01,False
7,7,5.791353,1.634222e-02,False
8,8,0.000029,9.956853e-01,False
9,9,0.035464,8.506767e-01,False


In [24]:
fig = px.bar(df, x='feature', y='F', color='significant', log_y=True)
fig.update_xaxes(type='category', categoryorder='total descending')
fig.show()

In [25]:
from sklearn.manifold import TSNE

x_viz = TSNE(n_components=2).fit_transform(xts)
fig = px.scatter(x=x_viz[:,0], y=x_viz[:,1], color=y)

ValueError: n_components=2 must be between 1 and min(n_samples, n_features)=1 with svd_solver='randomized'

In [ ]:
from sklearn.metrics import get_scorer

scorer = get_scorer(paradigm.scoring)
scorer([0,1],[0,1])